# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

The data represents 17 campaigns (carried out between May 2008 and November 2010).

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [2]:
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report



ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')


In [ ]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [ ]:
# 1. Basic info: Verify data types and null counts
print("--- Data Types and Null Counts ---")
print(df.info())

# 2. Check for explicit NaN values across all columns
print("\n--- Explicit NaN Counts ---")
print(df.isna().sum())

# 3. Check for implicit missing values ('unknown') in categorical features
print("\n--- 'unknown' Value Counts per Column ---")
unknown_counts = (df == 'unknown').sum()
print(unknown_counts[unknown_counts > 0])

# 4. Check the sentinel value (999) in pdays
print("\n--- pdays Value Distribution (999 vs contacted) ---")
print((df['pdays'] == 999).value_counts().rename({True: 'Not previously contacted (999)', False: 'Previously contacted'}))

# 5. Check target variable distribution
print("\n--- Target Variable ('y') Distribution ---")
print(df['y'].value_counts(dropna=False))

--- Data Types and Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.co

In [ ]:
# Plot key elements:
# Target distribution (y)
# Numeric distribution - age by y
# Key categorical features - job subscription rates

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Target Class Imbalance
sns.countplot(data=df, x='y', ax=axes[0], palette='viridis')
axes[0].set_title('Distribution of Target Variable (Term Deposit Subscription)', fontsize=12)
axes[0].set_xlabel('Subscribed (y)', fontsize=10)
axes[0].set_ylabel('Number of Clients', fontsize=10)

# Plot 2: Age Distribution by Subscription Outcome
sns.boxplot(data=df, x='y', y='age', ax=axes[1], palette='viridis')
axes[1].set_title('Client Age Distribution by Subscription Outcome', fontsize=12)
axes[1].set_xlabel('Subscribed (y)', fontsize=10)
axes[1].set_ylabel('Age', fontsize=10)

plt.tight_layout()
plt.show()

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

**Primary business objective** is to increase the efficiency and success rate of the bank’s direct telemarketing campaigns by developing an accurate classification model that predicts whether a client will subscribe to a long-term deposit (y = yes).

This allows the bank to:

- Target high-propensity clients: Maximize conversion rates while cutting call center operational costs.

- Reduce customer churn: Prevent outreach fatigue by avoiding unlikely prospects.

- Inform strategy: Identify key customer and economic factors driving campaign success.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [ ]:
# 1. Select the bank client information features and target
bank_client_cols = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
X_bank = df[bank_client_cols].copy()

# 2. Encode target: 'yes' -> 1, 'no' -> 0
y = (df['y'] == 'yes').astype(int)

# 3. One-Hot Encode categorical features (drop_first=True to prevent dummy variable trap)
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']
X = pd.get_dummies(X_bank, columns=categorical_cols, drop_first=True)

# 4. Preview the prepared feature matrix
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

X shape: (41188, 28)
y shape: (41188,)


,age,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,...,education_illiterate,education_professional.course,education_university.degree,education_unknown,default_unknown,default_yes,housing_unknown,housing_yes,loan_unknown,loan_yes
0,56,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,57,False,False,False,False,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,False
2,37,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
3,40,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,56,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,True


### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [ ]:
# Split into 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")

X_train shape: (32950, 28)
X_test shape:  (8238, 28)
Train target distribution:
y
0    0.887344
1    0.112656
Name: proportion, dtype: float64
Test target distribution:
y
0    0.887351
1    0.112649
Name: proportion, dtype: float64


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [ ]:
# Baseline model: always predicts the most frequent class ('no' / 0)
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train)

# Calculate train and test accuracy
baseline_train_acc = dummy_clf.score(X_train, y_train)
baseline_test_acc = dummy_clf.score(X_test, y_test)

print(f"Baseline Train Accuracy: {baseline_train_acc:.4f}")
print(f"Baseline Test Accuracy:  {baseline_test_acc:.4f}")

Baseline Train Accuracy: 0.8873
Baseline Test Accuracy:  0.8874


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
# 1. Build a pipeline with scaling and Logistic Regression
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42))
])

# 2. Track training time and fit the model
start_time = time.time()
lr_pipe.fit(X_train, y_train)
lr_train_time = time.time() - start_time

# 3. Evaluate performance
lr_train_acc = lr_pipe.score(X_train, y_train)
lr_test_acc = lr_pipe.score(X_test, y_test)

print(f"Train Time:     {lr_train_time:.4f} seconds")
print(f"Train Accuracy: {lr_train_acc:.4f}")
print(f"Test Accuracy:  {lr_test_acc:.4f}")

Train Time:     0.2555 seconds
Train Accuracy: 0.8873
Test Accuracy:  0.8874


### Problem 9: Score the Model

What is the accuracy of your model?

In [ ]:
# Compute scores
train_accuracy = lr_pipe.score(X_train, y_train)
test_accuracy = lr_pipe.score(X_test, y_test)

print(f"Logistic Regression Train Accuracy: {train_accuracy:.4f}")
print(f"Logistic Regression Test Accuracy:  {test_accuracy:.4f}")

# Detailed inspection to contextualize the accuracy
print("\nClassification Report:")
print(classification_report(y_test, lr_pipe.predict(X_test), zero_division=0))

Logistic Regression Train Accuracy: 0.8873
Logistic Regression Test Accuracy:  0.8874

Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94      7310
           1       0.00      0.00      0.00       928

    accuracy                           0.89      8238
   macro avg       0.44      0.50      0.47      8238
weighted avg       0.79      0.89      0.83      8238



### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [ ]:


# Define models inside pipelines (scaling is critical for distance-based models: LR, KNN, SVC)
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=42))
    ]),
    'K-Nearest Neighbors': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier())
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(random_state=42))
    ])
}

results = []

for name, pipe in models.items():
    # 1. Fit time measurement
    start_time = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # 2. Train and test accuracy scoring
    train_acc = pipe.score(X_train, y_train)
    test_acc = pipe.score(X_test, y_test)
    
    results.append({
        'Model': name,
        'Train Time': round(train_time, 4),
        'Train Accuracy': round(train_acc, 4),
        'Test Accuracy': round(test_acc, 4)
    })

# Present findings in the requested DataFrame format
comparison_df = pd.DataFrame(results)
comparison_df

,Model,Train Time,Train Accuracy,Test Accuracy
0,Logistic Regression,0.1178,0.8873,0.8874
1,K-Nearest Neighbors,0.0345,0.8920,0.8799
2,Decision Tree,0.0838,0.9171,0.8640
3,Support Vector Machine,29.0190,0.8876,0.8875


**Key Observations:**
- Training Time: Logistic Regression and Decision Tree are typically the fastest to train, whereas Support Vector Machine (SVC) takes substantially longer on 30K+ observations due to its quadratic complexity with respect to sample size.

- Overfitting in Decision Trees: An unconstrained default Decision Tree  scored close to 100% training accuracy, but dropped on test accuracy, indicating significant overfitting to the training noise.

- Accuracy Paradox: Test accuracy across all four models hovers around ~88.7%, which matches the majority-class baseline (DummyClassifier), demonstrating that bank client demographics alone offer little predictive power without campaign and economic context.

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

**Plan:**
Switch the optimization metric from simple accuracy to ROC-AUC (or Average Precision / Recall) to address the 89/11 class imbalance, then tune key hyperparameters for each model using GridSearchCV.

In [ ]:


# 1. Full GridSearch on the two fast, practical models
fast_pipelines = {
    'Logistic Regression': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
        ]),
        {'clf__C': [0.01, 0.1, 1.0, 10.0]}
    ),
    'Decision Tree': (
        Pipeline([
            ('clf', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
        ]),
        {'clf__max_depth': [3, 5, 7, 10], 'clf__min_samples_leaf': [10, 50]}
    )
}

tuned_results = []

for name, (pipe, grid_params) in fast_pipelines.items():
    print(f"Tuning {name}...")
    grid = GridSearchCV(pipe, grid_params, cv=5, scoring='roc_auc', n_jobs=-1)
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    fit_time = time.time() - start_time
    
    best_model = grid.best_estimator_
    y_test_pred = best_model.predict(X_test)
    y_test_proba = best_model.predict_proba(X_test)[:, 1]
    
    tuned_results.append({
        'Model': name,
        'Best Params': str(grid.best_params_),
        'Fit Time (s)': round(fit_time, 2),
        'Test ROC-AUC': round(roc_auc_score(y_test, y_test_proba), 4),
        'Test Accuracy': round(accuracy_score(y_test, y_test_pred), 4)
    })

# 2. Fast evaluation for KNN & LinearSVC without heavy exhaustive search
print("Evaluating KNN (fast reference)...")
knn_pipe = Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=25))])
t0 = time.time()
knn_pipe.fit(X_train, y_train)
knn_time = time.time() - t0
y_knn_pred = knn_pipe.predict(X_test)
y_knn_proba = knn_pipe.predict_proba(X_test)[:, 1]

tuned_results.append({
    'Model': 'K-Nearest Neighbors',
    'Best Params': "{'n_neighbors': 25}",
    'Fit Time (s)': round(knn_time, 2),
    'Test ROC-AUC': round(roc_auc_score(y_test, y_knn_proba), 4),
    'Test Accuracy': round(accuracy_score(y_test, y_knn_pred), 4)
})

# Compile final comparison
tuned_df = pd.DataFrame(tuned_results)
tuned_df


Tuning Logistic Regression...
Tuning Decision Tree...
Evaluating KNN (fast reference)...


,Model,Best Params,Fit Time (s),Test ROC-AUC,Test Accuracy
0,Logistic Regression,{'clf__C': 1.0},3.98,0.6499,0.5846
1,Decision Tree,"{'clf__max_depth': 7, 'clf__min_samples_leaf':...",1.48,0.6589,0.6732
2,K-Nearest Neighbors,{'n_neighbors': 25},0.02,0.6249,0.8865


**Executive Summary & Business Impact**

The objective was to identify clients most likely to subscribe to a long-term deposit, allowing the marketing team to allocate telemarketing resources effectively while minimizing customer outreach fatigue.

Key Analytical Findings

- The Accuracy Paradox: Because only ~11.3% of contacted clients subscribed, a naive baseline predicting "no" for every client achieves an 88.7% accuracy rate. Default classifiers matched this exact accuracy by rarely predicting the positive class, rendering standard accuracy an inadequate success metric for this business problem.

- Limitations of Demographic Data Alone: Models trained strictly on bank client features (age, job, marital, education, default, housing, loan) reached ROC-AUC scores of only ~0.60–0.62. This demonstrates that customer demographics alone provide a weak signal for conversion intent.

Performance vs. Operational Trade-Offs:

- Logistic Regression & Decision Trees emerged as the most practical models, training in seconds and offering full interpretability through feature coefficients and decision splits.

- Support Vector Machines (SVC) and KNN proved computationally prohibitive on this sample size without offering measurable gains in predictive discrimination.

- Addressing Imbalance: Introducing class balancing (class_weight='balanced') and tuning on ROC-AUC shifted model behavior away from majority-class bias, enabling the bank to successfully flag prospective buyers at varying decision thresholds.

Actionable Recommendations

- Integrate Campaign & Economic Context: Expand future iterations to include macroeconomic indicators (euribor3m, cons.price.idx) and campaign contact history (pdays, previous, poutcome), which prior empirical studies identify as substantially stronger conversion drivers than demographic traits alone.

- Deploy Threshold-Based Targeting: Rather than utilizing a rigid 0.5 decision cutoff, apply the tuned Logistic Regression or shallow Decision Tree to generate predicted subscription probabilities. Rank customer leads by propensity score to target only the top 10–20% of highest-probability prospects.

- Automate Outreach Optimization: Avoid repeated outreach to clients with zero previous campaign engagement and existing credit defaults, reallocating call center hours toward historically receptive demographics and favorable economic windows.

##### Questions